In [ ]:
import sys
import matplotlib
import numpy
import pandas
import sklearn

print(
    sys.version.split()[0],
    numpy.__version__,
    pandas.__version__,
    sklearn.__version__,
    matplotlib.__version__)


3.13.0 2.3.4 2.3.3 1.7.2 3.10.7


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2305

data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]

X = data[feature_names].to_numpy()
y = data["spam"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print(
    X.shape,
    round(100 * y.mean(), 1),
    X_train.shape,
    X_test.shape
)


(4204, 57) 39.9 (3153, 57) (1051, 57)


In [ ]:
data = pd.read_csv("data/spambase.csv")

print("Shape:", data.shape)
print("Spam percentage:", round(data["spam"].mean() * 100, 1))
print(data.head())

Shape: (4204, 58)
Spam percentage: 39.9
   word_freq_make  word_freq_address  word_freq_all  word_freq_3d  \
0             0.0               0.00           0.00           0.0   
1             0.0               0.00           0.00           0.0   
2             0.0               1.08           0.00           0.0   
3             0.0               0.00           0.00           0.0   
4             0.0               0.00           0.25           0.0   

   word_freq_our  word_freq_over  word_freq_remove  word_freq_internet  \
0           0.00            0.00               0.0                0.00   
1           0.84            0.00               0.0                0.00   
2           0.00            0.00               0.0                0.00   
3           2.04            0.00               0.0                0.00   
4           0.12            0.12               0.0                0.12   

   word_freq_order  word_freq_mail  ...  char_freq_semicolon  char_freq_paren  \
0             0.00 

In [11]:
## Part 1 — Bagging & OOB

def oob_mask(sample_indices, n_samples):
    """Return a boolean mask for rows absent from the bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask


In [ ]:
print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))


[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [ ]:
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    RandomForestClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)

bag = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    random_state=SEED,
    n_jobs=-1
)

lr = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

bag_lr = BaggingClassifier(
    lr,
    n_estimators=50,
    random_state=SEED
)

knn = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15)
)

ada = AdaBoostClassifier(
    n_estimators=400,
    random_state=SEED
)

members = [("lr", lr), ("knn", knn), ("tree", tree)]


In [14]:

bag.fit(X_train, y_train)

n = len(X_train)

oob_shares = []

for sample_indices in bag.estimators_samples_:
    mask = oob_mask(sample_indices, n)
    oob_shares.append(mask.mean())

mean_oob = np.mean(oob_shares)
mean_inbag = 1 - mean_oob

theoretical_oob = (1 - 1/n)**n
theoretical_inbag = 1 - theoretical_oob

print(f"Mean OOB share: {mean_oob:.6f}")
print(f"Mean in-bag share: {mean_inbag:.6f}")
print(f"Theoretical OOB share: {theoretical_oob:.6f}")
print(f"Theoretical in-bag share: {theoretical_inbag:.6f}")


Mean OOB share: 0.367431
Mean in-bag share: 0.632569
Theoretical OOB share: 0.367821
Theoretical in-bag share: 0.632179


In [15]:
# Number of training observations
n = len(X_train)

# Two columns: votes for class 0 and class 1
votes = np.zeros((n, 2), dtype=int)

for k, estimator in enumerate(bag.estimators_):
    # Find rows not used by this tree
    mask = oob_mask(bag.estimators_samples_[k], n)
    oob_indices = np.flatnonzero(mask)

    # Predict only OOB observations
    predictions = estimator.predict(X_train[mask])

    # Add votes for the predicted classes
    votes[oob_indices, predictions.astype(int)] += 1

# Choose the class with the most votes
# np.argmax selects class 0 in case of a tie
manual_predictions = np.argmax(votes, axis=1)

# Calculate accuracy
manual_oob_accuracy = np.mean(manual_predictions == y_train)

print(f"Manual OOB accuracy: {manual_oob_accuracy:.6f}")
print(f"Sklearn OOB accuracy: {bag.oob_score_:.6f}")
print(f"Absolute difference: {abs(manual_oob_accuracy - bag.oob_score_):.10f}")
print(f"Rows without OOB votes: {(votes.sum(axis=1) == 0).sum()}")


Manual OOB accuracy: 0.936886
Sklearn OOB accuracy: 0.936886
Absolute difference: 0.0000000000
Rows without OOB votes: 0


In [16]:
from sklearn.model_selection import cross_validate

models = {
    "Decision Tree": tree,
    "Bagging Tree": bag,
    "Random Forest": rf,
    "Logistic Regression": lr,
    "Bagging LR": bag_lr
}

cv_scores = {}

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )["test_score"]

    cv_scores[name] = scores

results = pd.DataFrame({
    name: scores for name, scores in cv_scores.items()
}).T

results.columns = [
    "Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"
]

results["Mean"] = results.mean(axis=1)
results["Std"] = results.iloc[:, :5].std(axis=1, ddof=0)

display(results.round(4))


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std
Decision Tree,0.8891,0.9065,0.8970,0.8968,0.8968,0.8972,0.0055
Bagging Tree,0.9319,0.9398,0.9271,0.9238,0.9254,0.9296,0.0058
Random Forest,0.9429,0.9604,0.9334,0.9429,0.9460,0.9451,0.0087
Logistic Regression,0.9461,0.9509,0.9319,0.9333,0.9397,0.9404,0.0073
Bagging LR,0.9398,0.9509,0.9287,0.9349,0.9381,0.9385,0.0073


In [17]:
comparisons = {
    "Bagging - Tree": cv_scores["Bagging Tree"] - cv_scores["Decision Tree"],
    "RF - Tree": cv_scores["Random Forest"] - cv_scores["Decision Tree"],
    "Bagging LR - LR": cv_scores["Bagging LR"] - cv_scores["Logistic Regression"]
}

paired_results = pd.DataFrame(comparisons).T

paired_results.columns = [
    "Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"
]

paired_results["Mean"] = paired_results.iloc[:, :5].mean(axis=1)
paired_results["Std"] = paired_results.iloc[:, :5].std(axis=1, ddof=0)

paired_results["Beyond spread"] = (
    paired_results["Mean"].abs() > paired_results["Std"]
)

display(paired_results.round(4))


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std,Beyond spread
Bagging - Tree,0.0428,0.0333,0.0301,0.0270,0.0286,0.0323,0.0056,True
RF - Tree,0.0539,0.0539,0.0365,0.0460,0.0492,0.0479,0.0064,True
Bagging LR - LR,-0.0063,0.0000,-0.0032,0.0016,-0.0016,-0.0019,0.0027,False


In [18]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# Fit on the full training split
rf.fit(X_train, y_train)

# OOB accuracy
rf_oob = rf.oob_score_

# Cross-validation accuracy
rf_cv_mean = cv_scores["Random Forest"].mean()
rf_cv_std = cv_scores["Random Forest"].std(ddof=0)

# Test accuracy
rf_test_predictions = rf.predict(X_test)
rf_test_accuracy = accuracy_score(y_test, rf_test_predictions)

# Standard error of test accuracy
n_test = len(y_test)

standard_error = np.sqrt(
    rf_test_accuracy * (1 - rf_test_accuracy) / n_test
)

# Agreement between OOB and CV
agreement = abs(rf_oob - rf_cv_mean) <= rf_cv_std

print(f"Random Forest OOB accuracy: {rf_oob:.6f}")
print(f"Random Forest CV accuracy: {rf_cv_mean:.6f} ± {rf_cv_std:.6f}")
print(f"Random Forest test accuracy: {rf_test_accuracy:.6f}")
print(f"Test standard error: {standard_error:.6f}")
print(f"OOB and CV agree: {agreement}")


Random Forest OOB accuracy: 0.949572
Random Forest CV accuracy: 0.945131 ± 0.008718
Random Forest test accuracy: 0.944814
Test standard error: 0.007043
OOB and CV agree: True


### Part 1 — Results and Discussion

The manually calculated out-of-bag accuracy for Bagging was 0.936886, exactly matching scikit-learn's `oob_score_`. No training observations were left without OOB votes.

Bagging improved the Decision Tree's mean cross-validation accuracy from 0.8972 to 0.9296. The paired improvement was 0.0323 ± 0.0056, exceeding the fold spread. Random Forest achieved a higher mean accuracy of 0.9451, with a paired improvement over the single Decision Tree of 0.0479 ± 0.0064. These improvements are consistent with ensemble methods reducing the variance of unstable decision trees.

In contrast, bagging Lo 
 gistic Regression changed mean accuracy from 0.9404 to 0.9385. The paired difference was −0.0019 ± 0.0027, which lies inside the fold spread. Therefore, the results do not establish a winner between Logistic Regression and Bagged Logistic Regression. Bagging has a smaller effect on relatively stable learners such as regularized Logistic Regression.

For Random Forest, the OOB accuracy was 0.949572, the cross-validation accuracy was 0.945131 ± 0.008718, and the test accuracy was 0.944814 with a standard error of 0.007043. The absolute difference between OOB and CV accuracy was approximately 0.004441, smaller than the CV standard deviation. Therefore, the OOB and CV estimates agree according to the assignment's criterion. The test accuracy is also close to both estimates.